# Guardrails and safety

Read the [guide](README.md). Run cells in order with the notebooks and apps groups. Offline replay uses real SDK and local protocol paths; it does not measure live model quality.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_local_guardrails

Predict the outcome, then run the [script](examples/01_local_guardrails.py).


In [ ]:
"""Local guardrails: Nimbus Tier 3 lesson.

Run: uv run python modules/13_guardrails/examples/01_local_guardrails.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from strands import Agent

from common.audit import AuditPolicy, quote_tool
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


def main():
    _, cfg = lesson_settings()
    policy = AuditPolicy(max_tools=1)
    agent = Agent(
        model=get_model(
            cfg,
            responses=[
                tool_message("quote_trip", {"destination": "LAX"}),
                "The trip exceeds this budget.",
            ],
        ),
        tools=[quote_tool()],
        hooks=[policy],
        callback_handler=None,
        retry_strategy=None,
    )
    result = agent(
        "Quote Los Angeles", invocation_state=dict(budget_cents=50000), limits={"turns": 3}
    )
    print("Denied attempts:", [r for r in policy.rows if r.get("blocked")])
    complete(result, cfg)


run_lesson(main, "modules/13_guardrails/examples/01_local_guardrails.py")

## 02_prompt_injection

Predict the outcome, then run the [script](examples/02_prompt_injection.py).


In [ ]:
"""Prompt injection: Nimbus Tier 3 lesson.

Run: uv run python modules/13_guardrails/examples/02_prompt_injection.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.safety import injection_trial


def main():
    _, cfg = lesson_settings()
    for hardened in (False, True):
        report = injection_trial(cfg, hardened=hardened)
        print("Boundary enabled:", hardened, report)
        if hardened:
            assert not report["tool_executed"]
    print("Mock mode forces an attempted attack; live model behavior may differ.")
    complete()


run_lesson(main, "modules/13_guardrails/examples/02_prompt_injection.py")

## 03_least_privilege

Predict the outcome, then run the [script](examples/03_least_privilege.py).


In [ ]:
"""Least privilege: Nimbus Tier 3 lesson.

Run: uv run python modules/13_guardrails/examples/03_least_privilege.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

import sqlite3
from importlib import import_module

from common.data import bookings_connection
from common.lessons import complete, lesson_settings
from common.pretty import run_cli


def main():
    lesson_settings()
    with bookings_connection() as db:
        try:
            db.execute("CREATE TABLE forbidden (id INTEGER)")
        except sqlite3.OperationalError:
            print("Read-only database rejected a write")
    import json

    from common.config import ROOT

    policy = json.loads((ROOT / "modules/13_guardrails/read_only_aws_policy.json").read_text())
    print("Scoped use_aws discovery policy:", policy)
    # Reuse the same reviewed read-only wrapper and Stubber/live path taught in Module 06.
    # Its application check rejects terminate_instances before any request can be sent.
    demo = import_module("modules.06_community_tools.examples.05_read_only_aws")
    demo.main()
    complete()


run_lesson(main, "modules/13_guardrails/examples/03_least_privilege.py")

## 04_limits

Predict the outcome, then run the [script](examples/04_limits.py).


In [ ]:
"""Limits: Nimbus Tier 3 lesson.

Run: uv run python modules/13_guardrails/examples/04_limits.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from strands import Agent, tool

from common.audit import AuditPolicy
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


def main():
    _, cfg = lesson_settings()

    @tool
    def ping() -> str:
        """Return a local fixed response."""
        return "pong"

    policy = AuditPolicy(allowed=("ping",), max_tools=1)
    agent = Agent(
        model=get_model(
            cfg, responses=[tool_message("ping", {}, "one"), tool_message("ping", {}, "two")]
        ),
        tools=[ping],
        hooks=[policy],
        callback_handler=None,
        retry_strategy=None,
    )
    result = agent("Check twice", limits={"turns": 2, "output_tokens": 200, "total_tokens": 1000})
    print("Stop reason:", result.stop_reason, "attempts:", policy.calls)
    print("Token limits apply at turn boundaries; use asyncio.timeout for a wall-clock bound.")
    complete(result, cfg)


run_lesson(main, "modules/13_guardrails/examples/04_limits.py")

## 05_bedrock_guardrails

Predict the outcome, then run the [script](examples/05_bedrock_guardrails.py).


In [ ]:
"""Bedrock guardrails: Nimbus Tier 3 lesson.

Run: uv run python modules/13_guardrails/examples/05_bedrock_guardrails.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_guarded_model
from common.pretty import run_cli
from common.safety import apply_grounding


def main():
    args, cfg = lesson_settings()
    # KEY IDEA: attach for model requests; supply grounding context to ApplyGuardrail.
    model = get_guarded_model(cfg)
    trace_keys = []

    def trace(**event):
        metadata = event.get("event", {}).get("metadata", {})
        if "trace" in metadata:
            trace_keys.extend(metadata["trace"].keys())

    result = Agent(model=model, callback_handler=trace, retry_strategy=None)(
        "State that Nimbus support handles travel questions.", limits={"turns": 2}
    )
    print("Provider trace categories (if supplied):", trace_keys)
    print("Grounding assessment:", apply_grounding(mock=args.mock))
    complete(result, cfg)


run_lesson(main, "modules/13_guardrails/examples/05_bedrock_guardrails.py")

## hardened_support

Predict the outcome, then run the [script](use_case/hardened_support.py).


In [ ]:
"""Dened support: Nimbus Tier 3 lesson.

Run: uv run python modules/13_guardrails/use_case/hardened_support.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from pathlib import Path
from tempfile import TemporaryDirectory

from common.approvals import ApprovalLedger, approval_agent, request_prompt, responses_for
from common.audit import AuditPolicy
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.safety import injection_trial


def main():
    _, cfg = lesson_settings()
    print("Injected document:", injection_trial(cfg, hardened=True))
    with TemporaryDirectory() as directory:
        ledger = ApprovalLedger(Path(directory) / "support.db")
        key = ledger.propose("CUS-001", "DEMO-001", "refund", 60000)
        audit = AuditPolicy(allowed=("apply_action",), max_tools=2)
        agent = approval_agent(cfg, ledger, key, hooks=[audit])
        result = agent(request_prompt(ledger, key), limits={"turns": 3})
        if result.stop_reason == "interrupt":
            ledger.decide(key, "CUS-001", "reject", supervisor="SUPERVISOR-DEMO")
            result = agent(responses_for(result), limits={"turns": 3})
        print("Final status:", ledger.get(key, "CUS-001")["status"], "audit:", audit.rows)
        complete(result, cfg)


run_lesson(main, "modules/13_guardrails/use_case/hardened_support.py")

## Practice

Complete the [exercises](README.md#exercises) and [quiz](quiz.md), then run the module tests.
